# 01 · Master inference across all images and hops

Re-runs the trained model over **every** cached image graph, **every** hop (graph
density 1–10) and **every** subgraph, producing a master prediction table with
identifiers (patient / image / hop). This underlies all downstream analyses.

Runtime: ~1–2 min on CPU (≈79k tiny graphs).

In [ ]:
# --- notebook preamble: locate the reusable analysis library ------------------
import os, sys, warnings
warnings.filterwarnings("ignore")
LIB = os.path.join(os.path.dirname(os.path.abspath("__file__")), "lib")
if not os.path.isdir(LIB):
    LIB = os.path.join(os.getcwd(), "lib")
sys.path.insert(0, LIB)

# _lib_raman defines ROOT / DATA / MODEL_PATH / META_PATH (absolute paths to the
# original project). If you moved the data, override them here, e.g.:
#   import _lib_raman as L; L.ROOT = "/path/to/Project_Raman"; L.DATA = ...
import _lib_raman as L
print("Project ROOT :", L.ROOT)
print("Graph DATA   :", L.DATA)
print("Model        :", L.MODEL_PATH)
print("Classes      :", L.CLASSES)


In [ ]:
import numpy as np, pandas as pd, json
np.random.seed(42)
rows, (miss, unexp) = L.run_full_inference(device='cpu', verbose=True)
df = pd.DataFrame(rows)
print('prediction rows:', len(df))
df.head()

## Merge clinical metadata + reproducible patient-level split (seed 42)

In [ ]:
meta = pd.read_csv(L.META_PATH)
cols = ['image_id','Sample.Number.internal','Klinik','Info','sex','Surgery.Type',
        'RANO.Resect','thisClass','thisScore','MGMTStatus','MGMTScore']
meta_s = meta[[c for c in cols if c in meta.columns]].drop_duplicates('image_id')\
            .rename(columns={'Sample.Number.internal':'patient_meta','Klinik':'center','Info':'round'})
m = df.merge(meta_s, on='image_id', how='left')

# patient-level stratified 70:30 split (seed 42)
pat = (meta.groupby('Sample.Number.internal')
         .agg(cls=('thisClass','first')).reset_index()
         .rename(columns={'Sample.Number.internal':'patient_id'}))
rng = np.random.RandomState(42); val_ids=[]
for cls, grp in pat.groupby('cls'):
    ids = grp['patient_id'].values.copy(); rng.shuffle(ids)
    val_ids += ids[:int(round(0.30*len(ids)))].tolist()
val_ids = set(val_ids)
m['split'] = np.where(m['patient_meta'].isin(val_ids), 'val', 'train')
os.makedirs('outputs', exist_ok=True)
m.to_csv('outputs/master_predictions_meta.csv', index=False)
print('saved outputs/master_predictions_meta.csv  rows:', len(m))
print('val patients:', len(val_ids), '| val rows:', (m.split=='val').sum())

The resulting held-out split contains **71 validation patients**, matching the
manuscript. This table (`outputs/master_predictions_meta.csv`) feeds notebooks 03–05.